In [2]:
import pandas as pd
import numpy as np 

In [3]:
sessions = pd.read_csv("sessions.csv")
events = pd.read_csv("events.csv")
orders = pd.read_csv("orders.csv")
order_items = pd.read_csv("order_items.csv")
products = pd.read_csv("products.csv")

In [4]:
sessions.columns

Index(['session_id', 'customer_id', 'start_time', 'device', 'source',
       'country'],
      dtype='object')

In [5]:
orders.columns

Index(['order_id', 'customer_id', 'order_time', 'payment_method',
       'discount_pct', 'subtotal_usd', 'total_usd', 'country', 'device',
       'source'],
      dtype='object')

In [6]:
events.columns

Index(['event_id', 'session_id', 'timestamp', 'event_type', 'product_id',
       'qty', 'cart_size', 'payment', 'discount_pct', 'amount_usd'],
      dtype='object')

In [7]:
order_items.columns

Index(['order_id', 'product_id', 'unit_price_usd', 'quantity',
       'line_total_usd'],
      dtype='object')

In [8]:
products.columns

Index(['product_id', 'category', 'name', 'price_usd', 'cost_usd',
       'margin_usd'],
      dtype='object')

In [11]:
sessions.head()
orders.head()
events.head()

,event_id,session_id,timestamp,event_type,product_id,qty,cart_size,payment,discount_pct,amount_usd
0,1,1,2021-12-27T00:08:36,page_view,93.0,NaN,NaN,NaN,NaN,NaN
1,2,1,2021-12-27T00:16:36,page_view,1005.0,NaN,NaN,NaN,NaN,NaN
2,3,1,2021-12-27T00:18:01,add_to_cart,1005.0,1.0,NaN,NaN,NaN,NaN
3,4,1,2021-12-27T00:45:36,page_view,918.0,NaN,NaN,NaN,NaN,NaN
4,5,1,2021-12-27T01:03:36,page_view,946.0,NaN,NaN,NaN,NaN,NaN


In [12]:
events["timestamp"] = pd.to_datetime(events["timestamp"])

In [13]:
events["event_type"].value_counts()


event_type
page_view      539343
add_to_cart    143126
checkout        44909
purchase        33580
Name: count, dtype: int64

In [14]:
session_start = (
    events.groupby("session_id")["timestamp"]
    .min()
    .reset_index(name="session_start_time")
)


In [15]:
purchase_time = (
    events[events["event_type"] == "purchase"]
    .groupby("session_id")["timestamp"]
    .min()
    .reset_index(name="purchase_time")
)


In [16]:
session_time = session_start.merge(
    purchase_time,
    on="session_id",
    how="inner"
)

session_time["purchase_time_hours"] = (
    session_time["purchase_time"] - session_time["session_start_time"]
).dt.total_seconds() / 3600


In [17]:
session_time.describe()


,session_id,session_start_time,purchase_time,purchase_time_hours
count,33580.000000,33580,33580,33580.000000
mean,60593.424509,2022-12-05 20:57:11.226742016,2022-12-05 22:17:57.397081600,1.346158
min,2.000000,2020-01-01 00:34:58,2020-01-01 01:10:58,0.033333
25%,30644.750000,2021-06-23 10:52:25.500000,2021-06-23 11:49:25.500000,0.866667
50%,60809.500000,2022-12-05 07:30:05,2022-12-05 08:43:05,1.350000
75%,90572.500000,2024-05-27 06:45:04,2024-05-27 07:34:34,1.800000
max,119999.000000,2025-10-31 21:54:41,2025-10-31 22:59:41,3.333333
std,34691.980549,NaN,NaN,0.623013


In [18]:
# total number of events per session
event_counts = (
    events.groupby("session_id")
    .size()
    .reset_index(name="total_events")
)

# number of add-to-cart events
add_to_cart_counts = (
    events[events["event_type"] == "add_to_cart"]
    .groupby("session_id")
    .size()
    .reset_index(name="add_to_cart_events")
)

# number of page views
page_view_counts = (
    events[events["event_type"] == "page_view"]
    .groupby("session_id")
    .size()
    .reset_index(name="page_view_events")
)


In [19]:
# total amount spent per session (from purchase event)
purchase_amount = (
    events[events["event_type"] == "purchase"]
    .groupby("session_id")["amount_usd"]
    .sum()
    .reset_index(name="total_amount_usd")
)


In [20]:
session_features = session_time.merge(event_counts, on="session_id")
session_features = session_features.merge(add_to_cart_counts, on="session_id", how="left")
session_features = session_features.merge(page_view_counts, on="session_id", how="left")
session_features = session_features.merge(purchase_amount, on="session_id", how="left")

session_features.fillna(0, inplace=True)


In [21]:
session_features.head()
session_features.describe()


,session_id,session_start_time,purchase_time,purchase_time_hours,total_events,add_to_cart_events,page_view_events,total_amount_usd
count,33580.000000,33580,33580,33580.000000,33580.000000,33580.000000,33580.000000,33580.000000
mean,60593.424509,2022-12-05 20:57:11.226742016,2022-12-05 22:17:57.397081600,1.346158,8.940739,1.761852,5.178886,133.806357
min,2.000000,2020-01-01 00:34:58,2020-01-01 01:10:58,0.033333,4.000000,1.000000,1.000000,2.800000
25%,30644.750000,2021-06-23 10:52:25.500000,2021-06-23 11:49:25.500000,0.866667,7.000000,1.000000,4.000000,40.300000
50%,60809.500000,2022-12-05 07:30:05,2022-12-05 08:43:05,1.350000,9.000000,1.000000,5.000000,86.460000
75%,90572.500000,2024-05-27 06:45:04,2024-05-27 07:34:34,1.800000,11.000000,2.000000,7.000000,174.270000
max,119999.000000,2025-10-31 21:54:41,2025-10-31 22:59:41,3.333333,17.000000,7.000000,8.000000,2984.580000
std,34691.980549,NaN,NaN,0.623013,2.639176,0.941490,2.089644,152.130611


# modeling start

In [22]:
df = session_features.copy()

X = df.drop(columns=[
    "session_id",
    "session_start_time",
    "purchase_time",
    "purchase_time_hours"
])

y = df["purchase_time_hours"]



In [23]:
import numpy as np

y_log = np.log1p(y)


In [24]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y_log, test_size=0.2, random_state=42
)


In [25]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_train, y_train)


LinearRegression()

In [26]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_pred_log = lr.predict(X_test)

y_pred = np.expm1(y_pred_log)
y_test_original = np.expm1(y_test)

mae = mean_absolute_error(y_test_original, y_pred)
rmse = np.sqrt(mean_squared_error(y_test_original, y_pred))
r2 = r2_score(y_test_original, y_pred)

print("Baseline Linear Regression")
print("MAE :", round(mae, 3))
print("RMSE:", round(rmse, 3))
print("R²  :", round(r2, 3))


Baseline Linear Regression
MAE : 0.25
RMSE: 0.319
R²  : 0.73


In [27]:
from xgboost import XGBRegressor

xgb = XGBRegressor(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb.fit(X_train, y_train)


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=300,
             n_jobs=None, num_parallel_tree=None, ...)

In [28]:
y_pred_log = xgb.predict(X_test)

y_pred = np.expm1(y_pred_log)
y_test_original = np.expm1(y_test)

mae = mean_absolute_error(y_test_original, y_pred)
rmse = np.sqrt(mean_squared_error(y_test_original, y_pred))
r2 = r2_score(y_test_original, y_pred)

print("XGBoost Results")
print("MAE :", round(mae, 3))
print("RMSE:", round(rmse, 3))
print("R²  :", round(r2, 3))


XGBoost Results
MAE : 0.245
RMSE: 0.314
R²  : 0.74


In [29]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(
    xgb,
    X,
    y_log,
    cv=5,
    scoring="r2"
)

print("CV R² scores:", cv_scores)
print("Mean CV R²:", cv_scores.mean())


CV R² scores: [0.78094422 0.778478   0.77544475 0.78002046 0.78056916]
Mean CV R²: 0.7790913184283524


In [31]:
import joblib
joblib.dump(xgb, "purchase_time_xgb.pkl")


['purchase_time_xgb.pkl']

In [32]:
feature_order = X.columns.tolist()
joblib.dump(feature_order, "feature_order.pkl")


['feature_order.pkl']

In [33]:
import joblib
import numpy as np
import pandas as pd

model = joblib.load("purchase_time_xgb.pkl")
feature_order = joblib.load("feature_order.pkl")

def predict_purchase_time(input_dict):
    """
    input_dict example:
    {
      "total_events": 10,
      "add_to_cart_events": 2,
      "page_view_events": 6,
      "total_amount_usd": 120.5
    }
    """
    X = pd.DataFrame([input_dict])[feature_order]
    y_log = model.predict(X)
    y_hours = np.expm1(y_log)
    return float(round(y_hours[0], 2))
